# Data Analysis Project Phase 1: Data Collection and MongoDB
**Name:** Austin Dachel

## Project Summary
This program downloads crash data in JSON format via HTTP, writes the data to MongoDB Atlas, saves the downloaded JSON to a local file, and verifies the number of records saved.

The original data source is NHTSA CrashAPI (FARS fatal crash-related data). Due to API access limits, the JSON files were mirrored into a GitHub repository and downloaded from GitHub raw URLs via HTTP.

## Configuration and Setup

- MongoDB Atlas connection information
- Database and collection names
- GitHub mirror repository location
- https://www.github.com/ausdac/crash-data-mirror/tree/main
- Years included in the dataset (2019–2023)
- File naming patterns for crash-level and vehicle-level JSON files
- Local directory where downloaded JSON files will be stored

In [1]:
import os
from pathlib import Path

# Mongo information & auth
MONGO_URI = os.getenv(
    "MONGO_URI",
    "mongodb+srv://austindachel_db_user:<redacted>@coloradomotofatalities.8s8a8nt.mongodb.net/?appName=ColoradoMotoFatalities"
)

# Database + collection names
DB_NAME = "ColoradoMotoFatalities"   
COLL_CRASHES = "co_fatal_crashes_2019_2023"
COLL_VEHICLES = "co_vehicles_2019_2023"

# Github mirror repo information
OWNER = "ausdac"
REPO = "crash-data-mirror"
BRANCH = "main"
DATA_DIR = "data"

# Years included for pulling data
YEARS = [2019, 2020, 2021, 2022, 2023]

# File naming pattern for the repo
CRASH_FILE = "colorado_fatal_crashes_{year}.json" # Crash level dataset
VEH_FILE   = "colorado_motorcycle_{year}.json" # Vehicle level dataset

# Write to a local file
OUT_DIR = Path("downloaded_json")
OUT_DIR.mkdir(exist_ok=True)


## Downloading JSON Data via HTTP

1. Builds raw GitHub URLs for each JSON file.
2. Downloads each file using the requests library.
3. Saves the downloaded content locally.
4. Computes file metadata (size, record count, SHA-256 hash).
5. Normalizes the JSON structure into a list of records.
6. Prints dataset summary statistics.

Data Collected:
- Fatal crash-level data (2019–2023)
- Motorcycle vehicle-level data (2019–2023)

In [2]:
import json
import hashlib
from datetime import datetime, timezone
import requests

# Raw github URL for the files stored in the mirror repository
def raw_url(filename: str) -> str:
    return f"https://raw.githubusercontent.com/{OWNER}/{REPO}/{BRANCH}/{DATA_DIR}/{filename}"

# Added in a compute for a SHA-256 checksum for the downloaded file
# Source: https://docs.python.org/3/library/hashlib.html
def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

# Normalize the different jsons into a simple list
def normalize_rows(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict) and "Results" in payload:
        res = payload["Results"]
        if res and isinstance(res[0], list):
            return res[0]
        return res
    raise TypeError(f"Unexpected JSON shape: {type(payload).__name__}")

# Download one json file, save it locally and return:
# data
# payload
# rows
def download_and_save(filename: str) -> dict:
    url = raw_url(filename)

    # json download
    r = requests.get(url, timeout=60)
    r.raise_for_status()

    content = r.content
    out_path = OUT_DIR / filename
    out_path.write_bytes(content)

    # parse json and normalize records
    payload = r.json()
    rows = normalize_rows(payload)

    # build metadata for reporting
    meta = {
        "filename": filename,
        "url": url,
        "saved_to": str(out_path),
        "bytes": len(content),
        "sha256": sha256_bytes(content),
        "records": len(rows),
        "downloadedAtUtc": datetime.now(timezone.utc).isoformat(),
    }
    return {"meta": meta, "payload": payload, "rows": rows}

# download all files
downloads = []
for y in YEARS:
    downloads.append(download_and_save(CRASH_FILE.format(year=y)))
    downloads.append(download_and_save(VEH_FILE.format(year=y)))

# print a quick dataset summary
for d in downloads:
    m = d["meta"]
    print(f'{m["filename"]}: records={m["records"]}, sizeMB={m["bytes"]/1024/1024:.2f}')


colorado_fatal_crashes_2019.json: records=543, sizeMB=0.09
colorado_motorcycle_2019.json: records=872, sizeMB=4.58
colorado_fatal_crashes_2020.json: records=570, sizeMB=0.09
colorado_motorcycle_2020.json: records=885, sizeMB=4.78
colorado_fatal_crashes_2021.json: records=637, sizeMB=0.10
colorado_motorcycle_2021.json: records=1024, sizeMB=5.52
colorado_fatal_crashes_2022.json: records=699, sizeMB=0.11
colorado_motorcycle_2022.json: records=1089, sizeMB=5.84
colorado_fatal_crashes_2023.json: records=664, sizeMB=0.11
colorado_motorcycle_2023.json: records=1010, sizeMB=5.47


In [3]:
# write a single json file containing all downloaded data
all_out = OUT_DIR / "co_phase1_all_downloads.json"
all_out.write_text(
    json.dumps(
        [{"meta": d["meta"], "payload": d["payload"]} for d in downloads],
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)
print("Wrote:", all_out)


Wrote: downloaded_json/co_phase1_all_downloads.json


## Writing Data to MongoDB Atlas

1. Connects to MongoDB Atlas using pymongo.
2. Clears the collections to allow reruns of the notebook.
3. Inserts crash-level records into:
   - co_fatal_crashes_2019_2023
4. Inserts vehicle-level records into:
   - co_vehicles_2019_2023
5. Adds a year field to each document for future phases

In [4]:
from pymongo import MongoClient

# connect to mongoDB atlas
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
crashes = db[COLL_CRASHES]
vehicles = db[COLL_VEHICLES]

# Make it rerunnable by clearing old data to avoid duplicates
crashes.delete_many({})
vehicles.delete_many({})

crash_inserted = 0
veh_inserted = 0

# insert into mongo
for d in downloads:
    filename = d["meta"]["filename"]
    rows = d["rows"]
    year = int(filename.split("_")[-1].split(".")[0])

    # extract year from the filename suffix
    if filename.startswith("colorado_fatal_crashes_"):
        # add a year field so its easier to query
        for r in rows:
            r["year"] = year
        if rows:
            crashes.insert_many(rows)
            crash_inserted += len(rows)

    elif filename.startswith("colorado_motorcycle_"):
        for r in rows:
            r["year"] = year
        if rows:
            vehicles.insert_many(rows)
            veh_inserted += len(rows)

print("Inserted crashes:", crash_inserted)
print("Inserted vehicles:", veh_inserted)

# Verification
print("\nCounts in MongoDB:")
print("Crash docs:", crashes.count_documents({}))
print("Vehicle docs:", vehicles.count_documents({}))

print("\nExample crash document:")
print(crashes.find_one({}, {"_id": 0}))

print("\nExample vehicle document:")
print(vehicles.find_one({}, {"_id": 0, "ST_CASE": 1, "VEH_NO": 1, "BODY_TYP": 1, "BODY_TYPNAME": 1, "DEATHS": 1, "year": 1}))


Inserted crashes: 3113
Inserted vehicles: 4880

Counts in MongoDB:
Crash docs: 3113
Vehicle docs: 4880

Example crash document:
{'CountyName': 'ADAMS (1)', 'CrashDate': '/Date(1546518000000-0500)/', 'Fatals': 1, 'Peds': 0, 'Persons': 1, 'St_Case': 80002, 'State': 8, 'StateName': 'Colorado', 'TotalVehicles': 1, 'year': 2019}

Example vehicle document:
{'BODY_TYP': '84', 'BODY_TYPNAME': 'Motor Scooter', 'DEATHS': '1', 'ST_CASE': '80001', 'VEH_NO': '1', 'year': 2019}


# Project Assignment: Phase 2 - Data Modeling
**Name:** Austin Dachel

## Project Summary

The objective of Phase 2 is to extract, clean, and model only the data relevant to motorcycle fatal crashes from MongoDB into a structured SQLite database.

This project is intentionally limited to:
- Crashes that involve at least one motorcycle-type vehicle
- Motorcycle vehicle records associated with those crashes

Non-motorcycle crash data is excluded at extraction time to ensure the SQLite database reflects the defined analysis scope.

### Initialize Dependencies

Import required libraries for MongoDB connection, SQLite modeling, date parsing, and validation utilities.

In [19]:
import re
import sqlite3
from datetime import datetime, timezone

SQLITE_PATH = "ColoradoMotoFatalities.sqlite"

MOTO_BODY_TYP = {80, 81, 82, 83, 84, 85}  # motorcycle-related body types

# Attempts to cast a value to an int
def safe_int(v):
    try:
        if v is None or v == "":
            return None
        return int(v)
    except (ValueError, TypeError):
        return None

# Standardizing string inputs
def safe_str(v, max_len=None):
    if v is None:
        return None
    s = str(v).strip()
    if not s:
        return None
    return s[:max_len] if max_len else s

"""
Parses strings like: "/Date(1546518000000-0500)/"
Returns 'YYYY-MM-DD' or None
"""
def parse_mongo_date_str(s):
    if not s:
        return None
    m = re.search(r"/Date\((\d+)([+-]\d{4})?\)/", str(s))
    if not m:
        return None
    ms = int(m.group(1))
    dt = datetime.fromtimestamp(ms / 1000, tz=timezone.utc)
    return dt.date().isoformat()

# Generate a unique primary key for the 'crash' table
def crash_id_from_crash(doc):
    y = safe_int(doc.get("year"))
    st_case = safe_int(doc.get("St_Case"))
    if y is None or st_case is None:
        return None
    return f"{y}-{st_case}"

# Generate a foreign key matching the 'crash' table
def crash_id_from_vehicle(doc):
    y = safe_int(doc.get("year"))
    st_case = safe_int(doc.get("ST_CASE"))
    if y is None or st_case is None:
        return None
    return f"{y}-{st_case}"

# Check if vehicle record matches defined motorcycle body types
def is_motorcycle_vehicle(vdoc):
    body = safe_int(vdoc.get("BODY_TYP"))
    return body in MOTO_BODY_TYP

### Build Motorcycle Crash ID Set

Collect unique crash IDs from vehicle records where BODY_TYP indicates a motorcycle (80–85).

In [20]:
# Building a set to filter the 'Crashes' collection effeciently later
moto_crash_ids = set()

for v in vehicles_col.find({}, {"year": 1, "ST_CASE": 1, "BODY_TYP": 1}):
    if is_motorcycle_vehicle(v):
        cid = crash_id_from_vehicle(v)
        if cid:
            moto_crash_ids.add(cid)

print("Unique motorcycle crash_ids found:", len(moto_crash_ids))

Unique motorcycle crash_ids found: 641


### Create SQLite Database Schema

Define relational tables for motorcycle crashes and vehicles, including primary and foreign key constraints.

In [ ]:
# Establish connections and enable FK constraints
conn = sqlite3.connect(SQLITE_PATH)
cur = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON;")

# Resetting table if needed for a clean run
cur.execute("DROP TABLE IF EXISTS moto_vehicles;")
cur.execute("DROP TABLE IF EXISTS moto_crashes;")

# Create master table for crash events
cur.execute("""
CREATE TABLE moto_crashes (
    crash_id        TEXT PRIMARY KEY,
    year            INTEGER,
    st_case         INTEGER,
    crash_date      TEXT,      -- YYYY-MM-DD
    county_name     TEXT,
    state           INTEGER,
    state_name      TEXT,
    fatals          INTEGER,
    persons         INTEGER,
    total_vehicles  INTEGER
);
""")

# Create detail table for vehicles involved in motorcycle crashes
cur.execute("""
CREATE TABLE moto_vehicles (
    vehicle_id      TEXT PRIMARY KEY,
    crash_id        TEXT NOT NULL,
    year            INTEGER,
    st_case         INTEGER,
    veh_no          INTEGER,
    body_typ        INTEGER,
    body_typ_name   TEXT,
    trav_sp         INTEGER,
    dr_drink        INTEGER,
    deaths          INTEGER,
    hour            INTEGER,
    minute          INTEGER,
    month           INTEGER,
    day             INTEGER,
    FOREIGN KEY (crash_id) REFERENCES moto_crashes(crash_id)
);
""")

# For time based analysis and relational lookups
cur.execute("CREATE INDEX IF NOT EXISTS idx_moto_crashes_year ON moto_crashes(year);")
cur.execute("CREATE INDEX IF NOT EXISTS idx_moto_vehicles_crash_id ON moto_vehicles(crash_id);")

conn.commit()

### Insert Filtered Crash Records

Insert only crash records associated with identified motorcycle crash IDs.

In [22]:
# Insert filtered crash records into the SQLite 'moto_crashes' table
crash_sql = """
INSERT OR REPLACE INTO moto_crashes
(crash_id, year, st_case, crash_date, county_name, state, state_name, fatals, persons, total_vehicles)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
"""

crash_inserted = 0
crash_skipped = 0

# Only insert if the crash is present in our motorcycle-only ID set
for c in crashes_col.find({}):
    cid = crash_id_from_crash(c)
    if not cid or cid not in moto_crash_ids:
        crash_skipped += 1
        continue

    row = (
        cid,
        safe_int(c.get("year")),
        safe_int(c.get("St_Case")),
        parse_mongo_date_str(c.get("CrashDate")),
        safe_str(c.get("CountyName"), 120),
        safe_int(c.get("State")),
        safe_str(c.get("StateName"), 60),
        safe_int(c.get("Fatals")),
        safe_int(c.get("Persons")),
        safe_int(c.get("TotalVehicles")),
    )
    cur.execute(crash_sql, row)
    crash_inserted += 1

conn.commit()
print("Inserted moto_crashes:", crash_inserted)
print("Skipped non-motorcycle crashes:", crash_skipped)

Inserted moto_crashes: 639
Skipped non-motorcycle crashes: 2474


### Insert Motorcycle Vehicle Records

Insert only motorcycle vehicle records while enforcing foreign key relationships.

In [23]:
# Insert motorcycle-specific vehicle data
veh_sql = """
INSERT OR REPLACE INTO moto_vehicles
(vehicle_id, crash_id, year, st_case, veh_no, body_typ, body_typ_name, trav_sp, dr_drink, deaths,
 hour, minute, month, day)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
"""

veh_inserted = 0
veh_skipped = 0

# Filtering for motorcycles only
for v in vehicles_col.find({}):
    if not is_motorcycle_vehicle(v):
        continue

    cid = crash_id_from_vehicle(v)
    if not cid:
        veh_skipped += 1
        continue

    # FK pre-check 
    cur.execute("SELECT 1 FROM moto_crashes WHERE crash_id = ? LIMIT 1;", (cid,))
    if cur.fetchone() is None:
        veh_skipped += 1
        continue

    y = safe_int(v.get("year"))
    st_case = safe_int(v.get("ST_CASE"))
    veh_no = safe_int(v.get("VEH_NO"))
    body_typ = safe_int(v.get("BODY_TYP"))

    vehicle_id = f"{cid}-{veh_no}" if veh_no is not None else safe_str(v.get("_id"))

    row = (
        vehicle_id,
        cid,
        y,
        st_case,
        veh_no,
        body_typ,
        safe_str(v.get("BODY_TYPNAME"), 80),
        safe_int(v.get("TRAV_SP")),
        safe_int(v.get("DR_DRINK")),
        safe_int(v.get("DEATHS")),
        safe_int(v.get("HOUR")),
        safe_int(v.get("MINUTE")),
        safe_int(v.get("MONTH")),
        safe_int(v.get("DAY")),
    )
    cur.execute(veh_sql, row)
    veh_inserted += 1

conn.commit()
print("Inserted moto_vehicles:", veh_inserted)
print("Skipped moto_vehicles (missing link/fields):", veh_skipped)

Inserted moto_vehicles: 660
Skipped moto_vehicles (missing link/fields): 3


### Verify Data Insertion

Print row counts to confirm successful extraction and modeling.

In [24]:
# Final validation
cur.execute("SELECT COUNT(*) FROM moto_crashes;")
print("moto_crashes row count:", cur.fetchone()[0])

cur.execute("SELECT COUNT(*) FROM moto_vehicles;")
print("moto_vehicles row count:", cur.fetchone()[0])

conn.close()

moto_crashes row count: 639
moto_vehicles row count: 660
